<a href="https://colab.research.google.com/github/Farrukh776/agentic-ai-hands-on/blob/main/harness_engineering/harness_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab: Building Agent Harnesses — From a Single Call to an Agent Loop

**Agentic AI — Hands-on Lab (Dr. Jawwad Ahmed Shamsi)**

In this lab you will build harnesses around the *same* small language model and measure how each design changes the score.
The model never changes — only the program around it (the **harness**).

**How this lab works**
* Tasks 1 → 9 go from easy to hard. Work in order and go **as far as you can**.
* Every task adds one row to your **score table**. Stopping after any task still leaves you with a complete, working result.
* Each task links to a section of the notes (*Agent Harnesses*, Sections 5 and 6).

**Before you start:** `Runtime → Change runtime type → T4 GPU`, then run the setup cells.

## 0. Setup

The next cell installs nothing new (Colab already has `transformers`) and loads **Qwen2.5-1.5B-Instruct**, a small open model.
It takes 2–3 minutes the first time. If no GPU is available, the cell switches to a smaller model automatically.

In [1]:
import torch, re, time, math, ast, operator, subprocess, tempfile, textwrap, string
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM

if torch.cuda.is_available():
    MODEL_NAME, DEVICE, DTYPE = "Qwen/Qwen2.5-1.5B-Instruct", "cuda", torch.float16
else:
    MODEL_NAME, DEVICE, DTYPE = "Qwen/Qwen2.5-0.5B-Instruct", "cpu", torch.float32
    print("No GPU found: using the smaller 0.5B model (slower and weaker).")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=DTYPE).to(DEVICE)
model.eval()
print("Loaded", MODEL_NAME, "on", DEVICE)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-1.5B-Instruct on cuda


### The only function that touches the model

`llm(prompt)` sends text to the model and returns its reply. **Everything else you write in this lab is harness.**
It also counts how many times the model is called, so we can compare the *cost* of each harness.

In [2]:
CALLS = 0

def llm(prompt, max_new_tokens=256):
    global CALLS
    CALLS += 1
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(llm("What is the capital of Pakistan? Reply in one word."))

Islamabad


### Questions, grader and score table

Run this cell. It defines the question sets, a **grader**, an **evaluation function** that prints an execution report, and the **score table**.

The grader is strict, like a real automatic grader: the harness must return **only the answer** (e.g. `224`, not `"Ali gets Rs. 224 back"`).

In [3]:
# ---------- Question set A: arithmetic word problems (Tasks 1-5) ----------
MATH = [
 ("Pens cost Rs. 37 each. Ali buys 48 pens and pays with a Rs. 2000 note. How much change does he get?", "224"),
 ("A train travels 245 km at a constant speed of 70 km per hour. How many minutes does the journey take?", "210"),
 ("A shop sells 3 notebooks for Rs. 125. How much do 27 notebooks cost?", "1125"),
 ("Ayesha saves Rs. 1850 every month. After 14 months she spends Rs. 9600 on a laptop. How much money does she have left?", "16300"),
 ("A class has 48 students and 3/8 of them are girls. How many boys are in the class?", "30"),
 ("A warehouse has 17 shelves with 23 boxes on each shelf, plus 389 boxes on the floor. How many boxes are there in total?", "780"),
 ("A school orders 127 packs of pencils with 49 pencils in each pack. It gives away 2000 pencils. How many pencils remain?", "4223"),
 ("A water tank holds 1200 litres and already contains 150 litres. Water flows in at 35 litres per minute. How many minutes until it is full?", "30"),
 ("A bus has 52 seats. What is the smallest number of buses needed to carry 1000 students?", "20"),
 ("A phone costs Rs. 4800. The shop gives a 15% discount, then adds 5% tax on the discounted price. What is the final price?", "4284"),
 ("What is the sum of all whole numbers from 1 to 150?", "11325"),
 ("A father is 3 times as old as his son. In 12 years he will be twice as old as his son. How old is the son now?", "12"),
]

# ---------- A small knowledge base (Tasks 6-9). The companies are fictional, ----------
# ---------- so the model cannot know the answers without reading the passages. ----------
PASSAGES = [
 "Indus Robotics is a company that builds warehouse robots. It was founded in 2009 by Sana Mirza and is headquartered in Hyderabad.",
 "Sana Mirza was born in Sukkur. She studied mechanical engineering before starting her own business.",
 "Thar Solar makes solar water pumps for farms. The company was founded in 2014 by Imran Baloch. Its main factory is in Multan.",
 "Imran Baloch was born in Gwadar and worked for ten years as an electrical engineer.",
 "The Kestrel K2 is a delivery drone made by Indus Robotics. It can carry up to 4 kilograms.",
 "The Kestrel K2 drone was first tested in 2021 at a farm near Larkana.",
 "Margalla Software develops accounting software for small shops. Its chief executive, Farah Qureshi, joined the company in 2016.",
 "Farah Qureshi grew up in Abbottabad and later worked at a bank in Lahore.",
 "The Ravi Prize is an award for engineering students. In 2023 it was won by Usman Tariq for a low-cost water filter.",
 "Usman Tariq is a student at a university in Peshawar. His water filter uses rice husk ash.",
 "Sindh Textiles operates a large spinning mill in Nawabshah. The mill employs 2300 workers.",
 "The Nawabshah mill of Sindh Textiles was opened in 1998 by the company's founder, Karim Shah.",
 "Karim Shah was born in Thatta and started his career as a cotton trader.",
 "Thar Solar's most popular product is the SunLift pump, which can lift water from a depth of 60 metres.",
 "Indus Robotics employs 340 people. Most of them work at the company's headquarters.",
 "Margalla Software's main product is called Hisaab. More than 50000 shops use it.",
 "Hyderabad is a city in Sindh known for its glass bangles.",
 "Multan is famous for its shrines and its mangoes.",
 "Many robotics startups in South Asia were founded after 2010.",
 "Solar pumps reduce the amount of diesel that farmers need to buy.",
]

SINGLE_HOP = [
 ("In which city is Indus Robotics headquartered?", "Hyderabad"),
 ("Who founded Thar Solar?", "Imran Baloch"),
 ("How many kilograms can the Kestrel K2 carry?", "4"),
 ("What is the name of Margalla Software's main product?", "Hisaab"),
 ("In which city is the main factory of Thar Solar?", "Multan"),
 ("How many workers does the Sindh Textiles mill employ?", "2300"),
]

TWO_HOP = [
 ("In which city was the founder of Indus Robotics born?", "Sukkur"),
 ("In which city was the founder of Thar Solar born?", "Gwadar"),
 ("Which company makes the aircraft that was first tested near Larkana?", "Indus Robotics"),
 ("In which city did the chief executive of Margalla Software grow up?", "Abbottabad"),
 ("In which city was the person who opened the Nawabshah mill born?", "Thatta"),
 ("In which city does the winner of the 2023 Ravi Prize study?", "Peshawar"),
]

CHALLENGE = [
 ("How many years after Indus Robotics was founded was Thar Solar founded?", "5"),
 ("How many more workers does the Sindh Textiles mill employ than Indus Robotics?", "1960"),
 ("How many years after the Nawabshah mill opened did Farah Qureshi join Margalla Software?", "18"),
]

# ---------- Grader ----------
def normalize(s):
    s = str(s).lower().strip()
    s = s.replace(",", "")                       # 1,860 -> 1860
    s = re.sub(r"\brs\.?\s*", "", s)             # drop currency
    s = s.strip(string.punctuation + " ")
    s = re.sub(r"^(the|a|an)\s+", "", s)
    s = re.sub(r"\.0+$", "", s)                  # 224.0 -> 224
    return s

def grade(prediction, gold):
    return 1 if normalize(prediction) == normalize(gold) else 0

# ---------- Evaluation with an execution report ----------
SCORES = []      # the score table: one row per harness

def evaluate(name, solve, questions, show=4):
    global CALLS
    CALLS = 0
    start, results = time.time(), []
    for i, (q, gold) in enumerate(questions, 1):
        try:
            pred = solve(q)
        except Exception as e:                   # a crashing harness scores 0, it does not stop the lab
            pred = f"[harness error: {e}]"
        results.append((q, gold, pred, grade(pred, gold)))
        print(f"Q{i}/{len(questions)} {'✓' if results[-1][3] else '✗'}", end="   ", flush=True)   # progress
    print()
    score = sum(r[3] for r in results) / len(results)
    calls = CALLS / len(questions)
    print(f"EXECUTION REPORT -- {name}")
    print(f"Score: {sum(r[3] for r in results)}/{len(results)} = {score:.2f}   "
          f"model calls per question: {calls:.1f}   time: {time.time()-start:.0f}s")
    failures = [r for r in results if r[3] == 0]
    for q, gold, pred, _ in failures[:show]:
        kind = "format" if normalize(gold) in normalize(pred) else "wrong"
        p = str(pred).replace("\n", " ")
        print(f"  [{kind:6}] expected {gold!r:16} got {p[:70]!r}")
    if len(failures) > show:
        print(f"  ... {len(failures)-show} more failures")
    SCORES.append((name, score, calls))
    return results

def show_scores():
    print(f"{'Harness':45} {'Score':>6} {'Calls/Q':>8}")
    for name, s, c in SCORES:
        print(f"{name:45} {s:6.2f} {c:8.1f}")

print(len(MATH), "math questions,", len(PASSAGES), "passages ready.")

12 math questions, 20 passages ready.


---
## Task 1 — Pattern 1: Single call *(Easy · Notes §6.1)*

```
Question ──► [Model] ──► Answer (raw text)
```
**Why this task.** Before improving anything you need a *baseline*: a score to compare every later harness against.

**The idea.** The simplest possible harness: send the question exactly as it is, return whatever the model replies.

**What to do.** You write no code — `solve_v1` is given.
1. Run the cell. `evaluate` sends all 12 maths problems to `solve_v1` one at a time and grades each reply. You will see a progress line (`Q1/12 ✗  Q2/12 …`) and then the execution report.
2. Look at one full reply: `print(r1[0][2])` (question 0, item 2 = what the harness returned).
3. Count the failure types with:
```python
fmt   = sum(1 for q, gold, pred, ok in r1 if not ok and normalize(gold) in normalize(pred))
wrong = sum(1 for q, gold, pred, ok in r1 if not ok) - fmt
print("format failures:", fmt, "  wrong answers:", wrong)
```

**Worked example (Question 1).**
* Question: *Pens cost Rs. 37 each. Ali buys 48 pens and pays with a Rs. 2000 note. How much change does he get?*
* Model reply (typical): *"To find the change, first find the total cost: 48 × 37 = 1776. Then 2000 − 1776 = 224. Ali gets Rs. 224 back."*
* Harness returns the whole paragraph → grader expected `224` → **wrong**, labelled `[format]` because 224 is inside the reply.

**What you should see.** A low score. Many failures are `[format]` (right answer, wrong form); some are `[wrong]` (real mistakes).

**What to record.** The score, the number of format failures, the number of wrong answers.

**Common mistakes.** Do not edit `solve_v1` to "fix" it — a low score is the point. Task 2 fixes it.

In [4]:
def solve_v1(question):
    return llm(question)

r1 = evaluate("T1 single call", solve_v1, MATH)

Q1/12 ✗   Q2/12 ✗   Q3/12 ✗   Q4/12 ✗   Q5/12 ✗   Q6/12 ✗   Q7/12 ✗   Q8/12 ✗   Q9/12 ✗   Q10/12 ✗   Q11/12 ✗   Q12/12 ✗   
EXECUTION REPORT -- T1 single call
Score: 0/12 = 0.00   model calls per question: 1.0   time: 131s
  [format] expected '224'            got 'To calculate the change Ali gets after buying 48 pens at Rs. 37 each w'
  [wrong ] expected '210'            got "To determine how long the train's journey takes in minutes, we can fol"
  [wrong ] expected '1125'           got 'To determine the cost of 27 notebooks when you know that 3 notebooks c'
  [format] expected '16300'          got 'To determine how much money Ayesha has left after spending Rs. 9600 on'
  ... 8 more failures


**Task 1 questions** (answer in this cell)
1. Look at the failures. How many are **format** failures (right number, wrong form) and how many are **wrong** answers?

   My score was 0/12. The execution report lists the first 4 failures: 2 were format failures (Q1 and Q4, where the right number was inside a long reply) and 2 were wrong answers (Q2 and Q3).

2. Print one full reply, e.g. `print(r1[0][2])`. Why does the grader reject it even when the number inside is correct?

   The model writes a long explanation, like "To calculate the change Ali gets after buying 48 pens…". The grader only accepts the short answer, so a full paragraph never equals "224", even when 224 is inside it. Long replies can also get cut off before the number is reached.

---
## Task 2 — Pattern 2: Single call with answer extraction *(Easy · Notes §6.2)*

```
Question ──► [Model: think, end with "Answer: …"] ──► [Harness: extract the value] ──► Answer
```
**Why this task.** In Task 1 many answers were correct but in the wrong *form*. The grader wants only the number.

**The idea.** Two small changes: (1) tell the model *where* to put its answer (a **format instruction**); (2) write a function that pulls out *only* the answer.

**What to do.**
1. **Format instruction.** In `solve_v2`, change `prompt = question` to
   `prompt = question + "\nThink step by step. On the last line write: Answer: <number only>"`
2. **Extraction.** Complete `extract_answer(reply)`:
   * if the reply contains `Answer:`, return the first number after it (remove commas: `1,860` → `1860`);
   * if there is no `Answer:` line, return the **last** number in the reply;
   * if there are no numbers at all, return the text after `Answer:` (needed later for answers that are names).

   Useful tools: `re.search(r"Answer:\s*(.+)", reply, re.IGNORECASE)` finds the text after `Answer:`;
   `re.findall(r"-?\d[\d,]*\.?\d*", text)` returns all numbers in `text`.
3. **Test your function** before evaluating:
```python
print(extract_answer("48 x 37 = 1776 ...\nAnswer: 224"))        # 224
print(extract_answer("The total is ...\nAnswer: 1,860 rupees"))  # 1860
print(extract_answer("so the answer is 66 apples"))               # 66
```
4. Run the evaluation cell.

**Worked example (Question 1).**
* Prompt: *Pens cost Rs. 37 each … How much change does he get?* + *Think step by step. On the last line write: Answer: <number only>*
* Model reply: *"48 × 37 = 1776. 2000 − 1776 = 224.  Answer: 224"*
* `extract_answer` returns `224` → **correct**.

**What you should see.** A large jump from Task 1 — with the **same model**. Remaining failures are mostly `[wrong]`.

**What to record.** The new score; are the remaining failures mostly format or wrong?

**Common mistakes.** Returning `"224 rupees"` instead of `"224"`; forgetting to remove commas; forgetting `return` (score 0, every answer `None`).

In [5]:
def extract_answer(reply):
    matches = re.findall(r"Answer:\s*(.+)", reply, re.IGNORECASE)
    if matches:
        after = matches[-1]
        nums = re.findall(r"-?\d[\d,]*\.?\d*", after)
        if nums:
            return nums[0].replace(",", "").rstrip(".")
        return after.strip().rstrip(".")
    nums = re.findall(r"-?\d[\d,]*\.?\d*", reply)
    if nums:
        return nums[-1].replace(",", "").rstrip(".")
    return reply.strip()

def solve_v2(question):
    prompt = question + "\nThink step by step. On the last line write: Answer: <number only>"
    reply = llm(prompt)
    return extract_answer(reply)

print(extract_answer("48 x 37 = 1776 ...\nAnswer: 224"))        # 224
print(extract_answer("The total is ...\nAnswer: 1,860 rupees"))  # 1860
print(extract_answer("so the answer is 66 apples"))               # 66

r2 = evaluate("T2 + answer extraction", solve_v2, MATH)

224
1860
66
Q1/12 ✓   Q2/12 ✓   Q3/12 ✗   Q4/12 ✓   Q5/12 ✗   Q6/12 ✗   Q7/12 ✗   Q8/12 ✓   Q9/12 ✓   Q10/12 ✓   Q11/12 ✓   Q12/12 ✓   
EXECUTION REPORT -- T2 + answer extraction
Score: 8/12 = 0.67   model calls per question: 1.0   time: 119s
  [wrong ] expected '1125'           got '4'
  [wrong ] expected '30'             got '18'
  [wrong ] expected '780'            got '389'
  [wrong ] expected '4223'           got '4303'


**Task 2 questions**
1. How much did the score change from Task 1? Did the *model* change?

   The score went from 0/12 (0.00) to 8/12 (0.67). No, the model did not change. I only added one line to the prompt and a small function that pulls out the number. This gave a big improvement at the same cost of 1 call per question.

2. Which failures remain? Are they mostly format or wrong answers now?

   The 4 remaining failures are all wrong answers, not format problems. For example, Q5 gave 18 (the number of girls) when it should give the boys, Q6 gave 389 (it ignored the shelves), and Q7 gave 4303 instead of 4223. These are real reasoning and calculation mistakes.

---
## Task 3 — Pattern 3: Prompt chain *(Easy · Notes §6.3)*

```
Question ──► [Call 1: list the numbers and the goal] ──facts──► [Call 2: solve using the facts] ──► [extract_answer] ──► Answer
```
**Why this task.** Some remaining mistakes come from *misreading* the problem (wrong number, wrong question).

**The idea.** Two model calls in a fixed order: call 1 only *understands* the problem, call 2 only *solves* it using the facts from call 1.

**What to do.** Complete `solve_v3(question)`:
1. Call 1: `facts = llm("List every number in this problem with its meaning, and state exactly what is asked. Do not solve it.\nProblem: " + question)`
2. Call 2: send a prompt containing the problem, the facts and the format instruction:
   `"Problem: " + question + "\nFacts:\n" + facts + "\nUsing these facts, solve the problem step by step. On the last line write: Answer: <number only>"`
3. Return `extract_answer(reply)` (your Task 2 function).

**Worked example (Question 5: 48 students, 3/8 are girls, how many boys?).**
* Call 1 reply: *"48 = total students; 3/8 = fraction who are girls. Asked: the number of boys."*
* Call 2 reply: *"Girls = 48 × 3/8 = 18. Boys = 48 − 18 = 30.  Answer: 30"* → harness returns `30`.

**What you should see.** The score may go up a little, stay the same, or even drop. What *always* happens: model calls per question double (1 → 2).

**What to record.** Score **and** calls per question, compared with Task 2. Is the extra cost worth it?

**Common mistakes.** Not including the original question in call 2; forgetting the format instruction in call 2.

In [6]:
def solve_v3(question):
    facts = llm("List every number in this problem with its meaning, and state exactly what is asked. Do not solve it.\nProblem: " + question)
    prompt = ("Problem: " + question + "\nFacts:\n" + facts +
              "\nUsing these facts, solve the problem step by step. On the last line write: Answer: <number only>")
    reply = llm(prompt)
    return extract_answer(reply)

r3 = evaluate("T3 prompt chain", solve_v3, MATH)

Q1/12 ✓   Q2/12 ✓   Q3/12 ✗   Q4/12 ✓   Q5/12 ✗   Q6/12 ✗   Q7/12 ✗   Q8/12 ✓   Q9/12 ✗   Q10/12 ✗   Q11/12 ✓   Q12/12 ✓   
EXECUTION REPORT -- T3 prompt chain
Score: 6/12 = 0.50   model calls per question: 2.0   time: 154s
  [wrong ] expected '1125'           got '1140.89'
  [wrong ] expected '30'             got '18'
  [wrong ] expected '780'            got '391'
  [wrong ] expected '4223'           got '4253'
  ... 2 more failures


**Task 3 questions**
1. Did splitting the work into two calls help? Compare the score **and** the model calls per question with Task 2.
2. When might a prompt chain hurt? (Hint: what happens if call 1 makes a mistake?)

Answers

1) No, it did not help. The score dropped from 8/12 (0.67) to 6/12 (0.50), and the calls per question doubled from 1.0 to 2.0. Q9 and Q10 were right in Task 2 but wrong here. So the extra cost was not worth it.
2) A prompt chain can hurt when call 1 makes a mistake. Call 2 trusts the facts from call 1, so a wrong or missing fact is passed forward and nothing checks it. For example, Q5 gave 18 in both Task 2 and Task 3, so the chain did not fix the misreading.

---
## Task 4 — Build the agent loop *(Medium · Notes §5 and §6.4)*

```
Question ──► [Model: one line] ──► CALC or FINAL? ──FINAL──► Answer
                    ▲                    │CALC
                    │                    ▼
           [Append RESULT] ◄── [Harness runs calculator]
```
**Why this task.** Small models make arithmetic slips (e.g. 127 × 49). A calculator never does.

**The idea.** Give the model a **tool**. The model cannot run the calculator — it can only *ask* by writing a line in an agreed format. The harness reads the line, runs the calculator and writes the result back, until the model gives a final answer. This is the **agent loop** from Section 5 of the notes.

The protocol (already in `AGENT_PROMPT`):
* model writes `CALC: <expression>` → harness replies `RESULT: <value>`
* model writes `FINAL: <answer>` → harness stops and returns the answer

**Given to you:** `calculator(expr)`, `parse_action(reply)` (returns `("calc", expr)`, `("final", answer)` or `("none", reply)`), and the loop. `transcript` is the whole conversation so far — the model sees all of it on every call.

**What to do.** Replace the four `...` in `solve_v4`, one line each:
```python
kind, value = parse_action(reply)                        # TODO 1  parse the reply
if kind == "final":
    return value                                         # TODO 2  stop
elif kind == "calc":
    result = calculator(value)                           # TODO 3  run the tool
    transcript += f"CALC: {value}\nRESULT: {result}\n"   # TODO 4  feed back the observation
```
Then run the next cell (one question), and then the evaluation.

**Worked example — the transcript for Question 1:**
```
CALC: 37*48        <- MODEL    (call 1)
RESULT: 1776       <- HARNESS  (calculator ran)
CALC: 2000-1776    <- MODEL    (call 2)
RESULT: 224        <- HARNESS
FINAL: 224         <- MODEL    (call 3) -> harness returns "224"
```

**What you should see.** Fewer arithmetic mistakes, but *new* kinds of failure: the model ignores the protocol, or uses all 6 steps without `FINAL:` (→ `"no answer within budget"`). About 3–5 calls per question.

**What to record.** Score, calls per question, and one example of a new kind of failure (add `print(transcript)` just before `return value` to see one).

**Common mistakes.** Forgetting `\n` at the end of the appended lines; appending only `RESULT` without the `CALC` line.

In [7]:
# A safe calculator: evaluates + - * / // % ** and brackets only (never use plain eval on model output!)
OPS = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul, ast.Div: operator.truediv,
       ast.FloorDiv: operator.floordiv, ast.Mod: operator.mod, ast.Pow: operator.pow, ast.USub: operator.neg}

def calculator(expr):
    def ev(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)): return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in OPS: return OPS[type(node.op)](ev(node.left), ev(node.right))
        if isinstance(node, ast.UnaryOp) and type(node.op) in OPS: return OPS[type(node.op)](ev(node.operand))
        raise ValueError("unsupported expression")
    try:
        v = ev(ast.parse(expr.replace(",", "").replace("^", "**"), mode="eval").body)
        return str(int(v)) if float(v).is_integer() else str(round(v, 4))
    except Exception as e:
        return f"Error: could not evaluate '{expr}' ({e})"

AGENT_PROMPT = '''You solve maths word problems using a calculator.
Write exactly ONE line per turn, in one of these two forms:
CALC: <arithmetic expression>
FINAL: <number only>
After each CALC line I will reply with RESULT: <value>.

Example:
Question: 12 boxes hold 15 apples each. 20 apples are eaten. How many are left?
CALC: 12*15
RESULT: 180
CALC: 180-20
RESULT: 160
FINAL: 160

Question: '''

def parse_action(reply):
    # returns ("calc", expression) or ("final", answer) or ("none", reply)
    for line in reply.splitlines():
        line = line.strip()
        if line.upper().startswith("CALC:"):  return ("calc", line[5:].strip())
        if line.upper().startswith("FINAL:"): return ("final", line[6:].strip())
    return ("none", reply)

print(calculator("37*48"), calculator("2000-1776"), calculator("import os"))

1776 224 Error: could not evaluate 'import os' (invalid syntax (<unknown>, line 1))


In [8]:
DEBUG = False

def solve_v4(question, max_steps=6):
    transcript = AGENT_PROMPT + question + "\n"
    for step in range(max_steps):
        reply = llm(transcript, max_new_tokens=60)
        kind, value = parse_action(reply)                        # TODO 1
        if kind == "final":
            if DEBUG: print(transcript)
            return value                                         # TODO 2
        elif kind == "calc":
            result = calculator(value)                           # TODO 3
            transcript += f"CALC: {value}\nRESULT: {result}\n"   # TODO 4
        else:
            transcript += "Please reply with one line: CALC: ... or FINAL: ...\n"
    if DEBUG: print(transcript)
    return "no answer within budget"

# Watch one run before evaluating all questions:
DEBUG = True
print(solve_v4(MATH[0][0]))
DEBUG = False

You solve maths word problems using a calculator.
Write exactly ONE line per turn, in one of these two forms:
CALC: <arithmetic expression>
FINAL: <number only>
After each CALC line I will reply with RESULT: <value>.

Example:
Question: 12 boxes hold 15 apples each. 20 apples are eaten. How many are left?
CALC: 12*15
RESULT: 180
CALC: 180-20
RESULT: 160
FINAL: 160

Question: Pens cost Rs. 37 each. Ali buys 48 pens and pays with a Rs. 2000 note. How much change does he get?
CALC: 37*48
RESULT: 1776
CALC: 2000 - 1776
RESULT: 224
CALC: 224
RESULT: 224
CALC: 224
RESULT: 224
CALC: 224
RESULT: 224
CALC: 37*48
RESULT: 1776

no answer within budget


In [9]:
r4 = evaluate("T4 agent loop + calculator", solve_v4, MATH)

Q1/12 ✗   Q2/12 ✗   Q3/12 ✗   Q4/12 ✓   Q5/12 ✗   Q6/12 ✗   Q7/12 ✗   Q8/12 ✗   Q9/12 ✗   Q10/12 ✓   Q11/12 ✓   Q12/12 ✗   
EXECUTION REPORT -- T4 agent loop + calculator
Score: 3/12 = 0.25   model calls per question: 5.8   time: 81s
  [wrong ] expected '224'            got 'no answer within budget'
  [wrong ] expected '210'            got 'no answer within budget'
  [wrong ] expected '1125'           got 'no answer within budget'
  [wrong ] expected '30'             got 'no answer within budget'
  ... 5 more failures


**Task 4 questions**
1. Print the transcript of one question (add a `print(transcript)` before `return`). Mark which lines the **model** wrote and which the **harness** wrote.
2. Did the calculator fix the arithmetic errors? What new kinds of failure appeared (e.g. the model ignoring the protocol, running out of steps)?
3. How many model calls per question does the loop use compared with Task 2? Why?

*Your answers:*

1. The transcript I printed looks like this. The model wrote CALC: 37*48, then CALC: 2000 - 1776, then CALC: 224 again and again. The harness wrote the instructions, the question, and every RESULT: line. The model got 1776 and then 224 correctly, but it never wrote FINAL:. It used all 6 steps and the harness returned "no answer within budget".
2. The calculator itself gave correct results, but the score went down to 3/12 (0.25), so it did not help overall. The new failure is that the model does not follow the protocol properly. It does not know when to stop and keeps repeating the same CALC line. Most failures ended with "no answer within budget".
3. The loop used 5.8 calls per question, compared with 1.0 in Task 2. Each CALC step needs a new model call, and most questions used nearly all 6 steps because the model never wrote FINAL.

---
## Task 5 — Pattern 5: Code-interpreter loop with error feedback *(Medium · Notes §6.5)*

```
Question ──► [Model: write Python] ──► [Sandbox: run code] ──output──► Answer
                    ▲                          │error
                    └──── error message ◄──────┘
```
**Why this task.** The calculator does one operation at a time. Some problems need *logic*: rounding up (buses) or solving an equation (ages). Code can do both.

**The idea.** The model writes a whole Python program that prints the answer. The harness runs it in a **sandbox** (separate process, 5-second limit). If it crashes, the harness sends the **error message** back and asks for a fix — up to 3 attempts.

**Given to you:** `extract_code(reply)`, `run_python(code)` (returns the printed output, or text starting with `Error`), `CODE_PROMPT`.

**What to do.** In `solve_v5`, after `output = run_python(code)`:
1. **If it worked** (`not output.startswith("Error")`): return the last printed line, `output.splitlines()[-1].strip()`.
2. **If it failed:** add the code and the error to the prompt so the next attempt can fix it:
```python
prompt += ("\n\nYour previous code was:\n```python\n" + code + "\n```\n"
           "It failed with:\n" + output + "\nFix the code. Print ONLY the final number.")
```

**Worked example (Question 9: buses for 1000 students, 52 seats each).**
* Attempt 1 — model writes `print(math.ceil(1000 / 52))` → sandbox: `NameError: name 'math' is not defined`
* Harness adds the code and the error to the prompt and asks again.
* Attempt 2 — model writes `import math; print(math.ceil(1000 / 52))` → output `20` → **correct**. Two calls.

**What you should see.** Usually the best maths score, especially on Q9 (buses) and Q12 (ages). 1–3 calls per question.

**What to record.** The score. Then set `max_tries=1`, run again, and record that score too — the difference is what **error feedback** contributed.

**Common mistakes.** Returning the whole output when several lines were printed; adding the error but not the code to the prompt.

In [10]:
def extract_code(reply):
    m = re.search(r"```(?:python)?\s*\n(.*?)```", reply, re.DOTALL)
    return m.group(1) if m else reply

def run_python(code, timeout=5):
    with tempfile.NamedTemporaryFile("w", suffix=".py", delete=False) as f:
        f.write(code)
    try:
        r = subprocess.run(["python3", "-I", f.name], capture_output=True, text=True, timeout=timeout)
        if r.returncode != 0:
            return "Error:\n" + r.stderr.strip()[-600:]
        return r.stdout.strip()[:500]
    except subprocess.TimeoutExpired:
        return f"Error: the code did not finish within {timeout} seconds"

CODE_PROMPT = ("Write a short Python program that solves the problem below and prints ONLY the final "
               "number. Put the code in a ```python``` block.\nProblem: ")

print(run_python("print(2000 - 37*48)"), "|", run_python("print(1/0)").splitlines()[-1])

224 | ZeroDivisionError: division by zero


In [11]:
def solve_v5(question, max_tries=3):
    prompt = CODE_PROMPT + question
    output = ""
    for attempt in range(max_tries):
        code = extract_code(llm(prompt, max_new_tokens=300))
        output = run_python(code)
        if not output.strip():
            output = "Error: the program printed nothing"
        if not output.startswith("Error"):
            return output.splitlines()[-1].strip()
        prompt += ("\n\nYour previous code was:\n```python\n" + code + "\n```\n"
                   "It failed with:\n" + output + "\nFix the code. Print ONLY the final number.")
    return output

r5 = evaluate("T5 code interpreter + error feedback", solve_v5, MATH)

Q1/12 ✓   Q2/12 ✓   Q3/12 ✓   Q4/12 ✓   Q5/12 ✓   Q6/12 ✓   Q7/12 ✓   Q8/12 ✓   Q9/12 ✓   Q10/12 ✓   Q11/12 ✓   Q12/12 ✓   
EXECUTION REPORT -- T5 code interpreter + error feedback
Score: 12/12 = 1.00   model calls per question: 1.1   time: 76s


In [12]:
r5b = evaluate("T5b code interpreter, max_tries=1", lambda q: solve_v5(q, max_tries=1), MATH)

for i in (8, 11):
    print(MATH[i][0][:55], "| gold:", MATH[i][1],
          "| T2:", r2[i][2], "| T4:", r4[i][2], "| T5:", r5[i][2])

Q1/12 ✓   Q2/12 ✓   Q3/12 ✓   Q4/12 ✓   Q5/12 ✓   Q6/12 ✓   Q7/12 ✓   Q8/12 ✓   Q9/12 ✓   Q10/12 ✓   Q11/12 ✓   Q12/12 ✗   
EXECUTION REPORT -- T5b code interpreter, max_tries=1
Score: 11/12 = 0.92   model calls per question: 1.0   time: 54s
  [wrong ] expected '12'             got 'Error: Traceback (most recent call last):   File "/tmp/tmpjmwjwsys.py"'
A bus has 52 seats. What is the smallest number of buse | gold: 20 | T2: 20 | T4: no answer within budget | T5: 20
A father is 3 times as old as his son. In 12 years he w | gold: 12 | T2: 12 | T4: no answer within budget | T5: 12


**Task 5 questions**
1. Compare Tasks 2, 4 and 5. Which harness handles the bus question (Q9) and the age question (Q12) best, and why?
2. Set `max_tries=1` and re-run. How much of the gain came from **error feedback**?
3. `run_python` uses a separate process and a time limit. What could go wrong if the harness ran the model's code with plain `exec()`?

*Your answers:*

1. On Q9 (buses) and Q12 (ages), Task 2 and Task 5 both got them right, and Task 4 failed both ("no answer within budget"). Overall Task 5 is the best, with 12/12 against 8/12 for Task 2 and 3/12 for Task 4, using only 1.1 calls per question. Code works well because Python does exact maths, can round up with math.ceil, and can solve equations.
2. With max_tries=3 the score was 12/12 (1.00). With max_tries=1 it was 11/12 (0.92). So error feedback fixed 1 question (Q12, which crashed on the first try). It is a small gain, and it cost almost nothing (1.1 calls instead of 1.0).
3. With plain exec() the model's code would run inside my own notebook. It could freeze everything with an infinite loop, delete or read files, or use my secrets. A separate process with a time limit keeps the damage away from the harness.

---
## About the next questions: single-hop, two-hop and challenge

Tasks 6–9 use questions about **fictional companies** described in `PASSAGES` (20 short passages). They differ in **how many passages you must read** to answer:

| Question type | Used in | Example | How it is answered |
|---|---|---|---|
| **Single-hop** | Task 6 | *In which city is Indus Robotics headquartered?* | One passage: *"Indus Robotics … founded in 2009 by Sana Mirza … headquartered in Hyderabad."* → **Hyderabad** |
| **Two-hop** | Tasks 7–8 | *In which city was the founder of Indus Robotics born?* | Hop 1 finds the founder is **Sana Mirza**; hop 2 searches *"Sana Mirza born"* and finds *"Sana Mirza was born in Sukkur."* → **Sukkur** |
| **Challenge** | Task 9 | *How many years after Indus Robotics was founded was Thar Solar founded?* | Look up 2009 and 2014, then compute 2014 − 2009 → **5** |

A single search cannot answer a two-hop question, because the question does not contain the name *Sana Mirza* — the harness must use what hop 1 found to write the second search.

**Why fictional companies?** If they were real, the model could answer from memory and the harness would not matter. Because they are invented, the model *must* read the passages — which is why the closed-book baseline in Task 6 should score close to zero.

---
## Task 6 — Pattern 6: Retrieve-then-answer (RAG) *(Medium · Notes §6.6)*

```
Question ──► [Search tool: top 3 passages] ──passages──► [Model: answer using only the passages] ──► [extract_answer] ──► Answer
```
**Why this task.** These questions are about facts the model has never seen. No prompt or parser helps if the information is missing.

**The idea.** Before calling the model, the harness **searches** the passages and puts the best matches into the prompt. The model answers from what it reads, not from memory.

**Given to you:** `search(query, k=3)` returns the 3 passages sharing the most words with the query. Try `print(search("Who founded Thar Solar?"))`.

**What to do.**
1. Run the **closed-book** cell first (no search). Print a few answers: what does the model do when it does not know?
2. Complete `solve_v6(question)`:
   * `passages = search(question)`
   * prompt: `"Answer the question using ONLY these passages.\nPassages:\n" + passages + "\nQuestion: " + question + "\nGive the shortest possible answer (a name or a number). On the last line write: Answer: <answer>"`
   * `reply = llm(prompt)` and `return extract_answer(reply)`

**Worked example ("Who founded Thar Solar?").**
* Closed book: the model invents a confident name → wrong.
* With search, the prompt contains *"Thar Solar makes solar water pumps … founded in 2014 by Imran Baloch …"* (and 2 more passages).
* Model reply: *"Answer: Imran Baloch"* → `extract_answer` returns `Imran Baloch` → **correct**.

**What you should see.** Closed book close to 0 (the model *guesses*); with retrieval, a high score with one call per question.

**What to record.** Both scores. For one remaining failure, print `search(question)`: was the passage **not found** (retrieval failure) or **found but answered wrongly** (answering failure)?

**Common mistakes.** Forgetting to put the passages in the prompt; long answers like *"The founder is Imran Baloch"* (the strict grader rejects them).

In [13]:
STOP = set("a an the of in on at to for by is was are which who what how many much did does do with from and its it his her their that this up".split())

def tokens(text):
    return [w for w in re.findall(r"[a-z0-9]+", text.lower()) if w not in STOP]

DF = Counter(w for p in PASSAGES for w in set(tokens(p)))

def search(query, k=3):
    # returns the k passages that share the most (rare) words with the query
    q = set(tokens(query))
    scored = [(sum(math.log(1 + len(PASSAGES) / DF[w]) for w in q & set(tokens(p))), p) for p in PASSAGES]
    scored.sort(key=lambda x: -x[0])
    return "\n".join("- " + p for s, p in scored[:k] if s > 0)

def solve_closed_book(question):
    return extract_answer(llm(question + "\nAnswer in a few words. On the last line write: Answer: <answer>"))

print(search("Who founded Thar Solar?"))
r6a = evaluate("T6a closed book (no retrieval)", solve_closed_book, SINGLE_HOP)

- Thar Solar makes solar water pumps for farms. The company was founded in 2014 by Imran Baloch. Its main factory is in Multan.
- Thar Solar's most popular product is the SunLift pump, which can lift water from a depth of 60 metres.
- Indus Robotics is a company that builds warehouse robots. It was founded in 2009 by Sana Mirza and is headquartered in Hyderabad.
Q1/6 ✗   Q2/6 ✗   Q3/6 ✗   Q4/6 ✗   Q5/6 ✗   Q6/6 ✗   
EXECUTION REPORT -- T6a closed book (no retrieval)
Score: 0/6 = 0.00   model calls per question: 1.0   time: 6s
  [wrong ] expected 'Hyderabad'      got 'Chandigarh'
  [wrong ] expected 'Imran Baloch'   got 'Dr. Ravi Gupta and Dr. Suresh Chandra'
  [wrong ] expected '4'              got '10'
  [wrong ] expected 'Hisaab'         got "Margalla Software's main product is Margalla CRM"
  ... 2 more failures


In [14]:
def solve_v6(question):
    passages = search(question)
    prompt = ("Answer the question using ONLY these passages.\nPassages:\n" + passages +
              "\nQuestion: " + question +
              "\nGive the shortest possible answer (a name or a number). On the last line write: Answer: <answer>")
    reply = llm(prompt)
    return extract_answer(reply)

r6 = evaluate("T6 retrieve-then-answer", solve_v6, SINGLE_HOP)

Q1/6 ✓   Q2/6 ✓   Q3/6 ✓   Q4/6 ✓   Q5/6 ✓   Q6/6 ✓   
EXECUTION REPORT -- T6 retrieve-then-answer
Score: 6/6 = 1.00   model calls per question: 1.0   time: 7s


**Task 6 questions**
1. Compare the closed-book score with RAG. What did the closed-book model do when it did not know the answer?
2. For any remaining failure, print `search(question)`. Was it a **retrieval** failure (passage not found) or an **answering** failure?

*Your answers:*

1. Closed book scored 0/6 and RAG scored 6/6. When the closed-book model did not know the answer, it made up a confident but wrong one. For example, it said "Chandigarh" for the Indus Robotics headquarters and "Margalla CRM" for the product name. It never said "I don't know".
2. I had no failures with RAG (6/6), so there was nothing to diagnose. The search found the right passage for each question and the model answered correctly from it.

---
## Task 7 — Pattern 7: Multi-hop retrieval *(Hard · Notes §6.7)*

```
Question ──► [Search 1] ──► [Model: summarize] ──► [Model: next query] ──► [Search 2] ──► [Model: answer] ──► Answer
                 └─────────── (b) raw passages from hop 1 ──────────────────────────────┘
```
**Why this task.** Two-hop questions need a fact that the question does not mention, so one search cannot find it.

**The idea.** Search twice. After the first search, the model writes a *new* query using what it learned; the second search finds the missing fact.

**What to do.**
1. Run the baseline: your Task 6 harness on `TWO_HOP` (expect a low score).
2. Complete `solve_v7(question, use_raw_passages)` in five steps:
   1. `p1 = search(question)` — hop 1
   2. `note = llm(...)`: *"Passages: … Question: … In one sentence, state what these passages tell us that helps answer the question."*
   3. `q2 = llm(...)`: *"Question: … What we know: … Write ONE short search query to find the missing information. Reply with the query only."* — keep the first line: `q2.splitlines()[0]`
   4. `p2 = search(q2)` — hop 2
   5. Final call: evidence = `p1 + p2` if `use_raw_passages` else `note + p2`; ask for the answer with the `Answer:` line; return `extract_answer(reply)`.
3. Run the cell — it evaluates both versions: (a) summary only, (b) raw passages.

**Worked example ("In which city was the founder of Indus Robotics born?").**
* Search 1 finds *"Indus Robotics … founded in 2009 by Sana Mirza …"*
* Note: *"Indus Robotics was founded by Sana Mirza."*  →  next query: *"Sana Mirza birthplace"*
* Search 2 finds *"Sana Mirza was born in Sukkur."*  →  final answer `Sukkur` (correct). Three calls.

**What you should see.** Baseline low; two hops much better; version (b) often at least as good as (a), because a one-sentence summary can drop a needed detail.

**What to record.** The three scores (baseline, a, b), and one question where (a) and (b) differ — explain why.

**Common mistakes.** Using the whole `q2` reply as the query (keep the first line); forgetting to pass `p2` to the final call.

In [15]:
r7base = evaluate("T7 baseline: single-hop RAG on 2-hop Qs", solve_v6, TWO_HOP)

def solve_v7(question, use_raw_passages=True):
    p1 = search(question)
    note = llm("Passages:\n" + p1 + "\nQuestion: " + question +
               "\nIn one sentence, state what these passages tell us that helps answer the question.")
    q2 = llm("Question: " + question + "\nWhat we know: " + note +
             "\nWrite ONE short search query to find the missing information. Reply with the query only.")
    q2 = q2.strip().splitlines()[0] if q2.strip() else question
    p2 = search(q2)
    evidence = (p1 + "\n" + p2) if use_raw_passages else (note + "\n" + p2)
    prompt = ("Answer the question using ONLY this evidence.\nEvidence:\n" + evidence +
              "\nQuestion: " + question +
              "\nGive the shortest possible answer (a name or a number). On the last line write: Answer: <answer>")
    reply = llm(prompt)
    return extract_answer(reply)

r7a = evaluate("T7a multi-hop, summary only", lambda q: solve_v7(q, use_raw_passages=False), TWO_HOP)
r7b = evaluate("T7b multi-hop, raw passages", lambda q: solve_v7(q, use_raw_passages=True), TWO_HOP)

Q1/6 ✗   Q2/6 ✗   Q3/6 ✓   Q4/6 ✗   Q5/6 ✗   Q6/6 ✗   
EXECUTION REPORT -- T7 baseline: single-hop RAG on 2-hop Qs
Score: 1/6 = 0.17   model calls per question: 1.0   time: 6s
  [wrong ] expected 'Sukkur'         got 'Hyderabad'
  [wrong ] expected 'Gwadar'         got 'Pakistan'
  [wrong ] expected 'Abbottabad'     got 'Hyderabad'
  [wrong ] expected 'Thatta'         got 'Karachi'
  ... 1 more failures
Q1/6 ✗   Q2/6 ✗   Q3/6 ✗   Q4/6 ✗   Q5/6 ✗   Q6/6 ✗   
EXECUTION REPORT -- T7a multi-hop, summary only
Score: 0/6 = 0.00   model calls per question: 3.0   time: 19s
  [wrong ] expected 'Sukkur'         got 'Hyderabad'
  [wrong ] expected 'Gwadar'         got 'Multan'
  [wrong ] expected 'Indus Robotics' got 'Thar Solar'
  [wrong ] expected 'Abbottabad'     got 'Not mentioned'
  ... 2 more failures
Q1/6 ✓   Q2/6 ✓   Q3/6 ✗   Q4/6 ✗   Q5/6 ✗   Q6/6 ✗   
EXECUTION REPORT -- T7b multi-hop, raw passages
Score: 2/6 = 0.33   model calls per question: 3.0   time: 13s
  [wrong ] expected 'Indus 

In [16]:
print(search(TWO_HOP[0][0]))
print("-----")
for a, b in zip(r7a, r7b):
    if a[2] != b[2]:
        print(a[0], "| gold:", a[1], "| (a):", a[2], "| (b):", b[2])

- Indus Robotics is a company that builds warehouse robots. It was founded in 2009 by Sana Mirza and is headquartered in Hyderabad.
- The Kestrel K2 is a delivery drone made by Indus Robotics. It can carry up to 4 kilograms.
- Indus Robotics employs 340 people. Most of them work at the company's headquarters.
-----
In which city was the founder of Indus Robotics born? | gold: Sukkur | (a): Hyderabad | (b): Sukkur
In which city was the founder of Thar Solar born? | gold: Gwadar | (a): Multan | (b): Gwadar
In which city did the chief executive of Margalla Software grow up? | gold: Abbottabad | (a): Not mentioned | (b): Sukkur


**Task 7 questions**
1. Why does single-hop RAG fail on these questions? Print `search(TWO_HOP[0][0])` and explain.
2. Did giving the final call the **raw passages** help compared with the summary only? Explain with one example.

*Your answers:*

1. print(search(TWO_HOP[0][0])) returned three passages, all about Indus Robotics (the founding, the Kestrel drone, the employees). The passage "Sana Mirza was born in Sukkur" is missing. The question never says "Sana Mirza", so the search matches the rare words "Indus" and "Robotics", and the founder's birthplace passage does not get into the top 3. One search cannot find a fact that the question does not mention.
2. Yes, raw passages helped. The summary-only version scored 0/6 and the raw-passages version scored 2/6 (the baseline was 1/6). Example: "In which city was the founder of Indus Robotics born?" The summary version answered Hyderabad and the raw version answered Sukkur. My explanation is that the link "founded by Sana Mirza" is in the first passage. The one sentence summary dropped it, so the model could not connect the founder to Sana Mirza. The raw passages kept it. On the Margalla question both versions were still wrong, so even the better version is not reliable with such a small model.

---
## Task 8 — Pattern 8: Verify-then-retry *(Hard · Notes §6.8)*

```
Question ──► [Two-hop search] ──► [Model: draft] ──► Draft in evidence? ──yes──► Answer
                                                          │no
                                                          ▼
                                   [Search again: draft + question] ──► [Model: answer again] ──► Answer
```
**Why this task.** Sometimes the model answers with something that is *not in the evidence at all* — a guess. The harness can detect this cheaply.

**The idea.** Before returning, check: **does the answer appear in the retrieved passages?** If yes, accept. If no, search again (draft + question as the query), add the new passages, and ask once more.

**Given to you:** `supported(answer, evidence)` → `True` if the answer text appears in the evidence.

**What to do.** Write `solve_v8(question)`:
1. Copy the two-hop steps from `solve_v7` (raw-passage version), keeping `evidence = p1 + "\n" + p2`.
2. Get a draft answer from the evidence (same final prompt as Task 7).
3. `if not supported(draft, evidence):` add `search(draft + " " + question, k=4)` to the evidence and ask again.
4. Return the (possibly new) answer.

Tip: count how often the check triggers with a global counter (`RETRIES += 1` inside the `if`).

**Worked example ("In which city was the person who opened the Nawabshah mill born?").**
* Hop 1 finds that *Karim Shah* opened the mill, but the next query is poor and hop 2 misses his passage.
* Draft *"Nawabshah"* — appears in the evidence, so it is **accepted** (a wrong answer passes the check!).
* In another run the draft is *"Karachi"* — not in the evidence → search again → more passages → the model answers again.

**What you should see.** A small improvement at best, with extra calls on some questions. "Appears in the evidence" is not the same as "correct".

**What to record.** The score; how many times the check triggered; one way the check could *wrongly accept* and one way it could *wrongly reject*.

**Common mistakes.** Checking the draft against the question instead of the evidence; retrying forever (retry **once**).

In [20]:
def supported(answer, evidence):
    a = normalize(answer)
    return len(a) > 0 and a in normalize(evidence)

RETRIES = 0

def answer_from(question, evidence):
    prompt = ("Answer the question using ONLY this evidence.\nEvidence:\n" + evidence +
              "\nQuestion: " + question +
              "\nGive the shortest possible answer (a name or a number). On the last line write: Answer: <answer>")
    return extract_answer(llm(prompt))

def solve_v8(question):
    global RETRIES
    p1 = search(question)
    note = llm("Passages:\n" + p1 + "\nQuestion: " + question +
               "\nIn one sentence, state what these passages tell us that helps answer the question.")
    q2 = llm("Question: " + question + "\nWhat we know: " + note +
             "\nWrite ONE short search query to find the missing information. Reply with the query only.")
    q2 = q2.strip().splitlines()[0] if q2.strip() else question
    p2 = search(q2)
    evidence = p1 + "\n" + p2
    draft = answer_from(question, evidence)
    if not supported(draft, evidence):
        RETRIES += 1
        evidence += "\n" + search(draft + " " + question, k=4)
        draft = answer_from(question, evidence)
    return draft

RETRIES = 0
r8 = evaluate("T8 multi-hop + verify-then-retry", solve_v8, TWO_HOP)
print("verification triggered:", RETRIES, "times")

Q1/6 ✓   Q2/6 ✓   Q3/6 ✗   Q4/6 ✗   Q5/6 ✗   Q6/6 ✗   
EXECUTION REPORT -- T8 multi-hop + verify-then-retry
Score: 2/6 = 0.33   model calls per question: 3.0   time: 17s
  [wrong ] expected 'Indus Robotics' got 'Thar Solar'
  [wrong ] expected 'Abbottabad'     got 'Sukkur'
  [wrong ] expected 'Thatta'         got 'Sukkur'
  [wrong ] expected 'Peshawar'       got 'Hyderabad'
verification triggered: 0 times


**Task 8 questions**
1. How often did the verification step trigger? (Add a counter or a `print`.) Did the retry fix those cases?
2. Give one way this check could **wrongly accept** a bad answer, and one way it could **wrongly reject** a good one.

*Your answers:*

1. The check triggered 0 times out of 6 questions. Because it never triggered, no retry happened, so there was nothing to fix. The score stayed at 2/6 (0.33), the same as Task 7b, and the cost stayed at 3.0 calls per question. In this run the verify step added nothing.
2. The check can wrongly accept a bad answer, and my own run shows it. The model answered "Sukkur" for the Margalla Software question (the right answer is Abbottabad) and "Thar Solar" for the Larkana question. These words were in the retrieved passages, so the check accepted them. "Appears in the evidence" does not mean "correct". It can also wrongly reject a good answer when the model words it differently from the passage, for example "Indus Robotics Ltd" or a number written in words. The text would not match, so a correct answer would cause an unneeded retry

---
## Task 9 — Challenge: Planner–executor *(Challenge · Notes §6.9)*

```
Question ──► [Planner: list steps] ──► [Executor: LOOKUP via RAG, COMPUTE via calculator] ──► [Combiner] ──► Answer
```
**Why this task.** `CHALLENGE` questions need *several* facts *and* a calculation. No harness from Tasks 1–8 does both.

**The idea.** Three roles: a **planner** (one model call) writes the steps; an **executor** runs each step with the right tool; a **combiner** produces the final answer. Your earlier harnesses become tools.

**What to do.** Design `solve_v9` yourself. Suggested design:
1. **Planner:** ask the model to break the question into at most 3 steps, one per line, each starting with `LOOKUP:` (a fact to find) or `COMPUTE:` (a calculation).
2. **Executor:** for each `LOOKUP:` line call your Task 6 harness, e.g. `solve_v6("What is " + step + "?")`, and store the result.
3. **Combiner:** give the question and all facts to the model; ask for one expression on a line starting with `CALC:`; parse it with `parse_action` and run `calculator`.

**Worked example ("How many years after Indus Robotics was founded was Thar Solar founded?").**
```
Planner writes:  1. LOOKUP: the year Indus Robotics was founded
                 2. LOOKUP: the year Thar Solar was founded
                 3. COMPUTE: the difference between the two years
Executor:        solve_v6("What is the year Indus Robotics was founded?") -> 2009
                 solve_v6("What is the year Thar Solar was founded?")     -> 2014
Combiner writes: CALC: 2014 - 2009   -> calculator -> 5   (correct, about 4 calls)
```

**What you should see.** Hard for a small model — a partial score is a good result. Failures can come from any role.

**What to record.** Score, calls per question, and for one failure which role failed (planner, executor or combiner).

**Common mistakes.** Plan lines that start with numbers (`"1. LOOKUP: …"`) — strip the number before checking for `LOOKUP:`; letting the model do the arithmetic instead of the calculator.

In [18]:
def solve_v9(question, verbose=False):
    plan = llm("Break this question into at most 3 steps, one per line. Each line must start with "
               "LOOKUP: (a single fact to find) or COMPUTE: (a calculation using earlier results). "
               "Do not answer the question.\nQuestion: " + question, max_new_tokens=120)
    facts = []
    for line in plan.splitlines():
        line = line.replace("*", "")
        line = re.sub(r"^\s*\d+[\.\)]\s*", "", line).strip()     # strip "1. "
        if line.upper().startswith("LOOKUP:"):
            step = line[len("LOOKUP:"):].strip()
            result = solve_v6("What is " + step + "?")
            facts.append(f"{step}: {result}")
    fact_text = "\n".join(facts)
    reply = llm("Question: " + question + "\nFacts:\n" + fact_text +
                "\nWrite ONE arithmetic expression using only the numbers in the facts that answers the question. "
                "Reply with one line: CALC: <expression>", max_new_tokens=60)
    if verbose:
        print("PLAN:\n" + plan + "\nFACTS:\n" + fact_text + "\nCOMBINER:\n" + reply)
    kind, value = parse_action(reply)
    if kind == "calc":
        return calculator(value)
    return extract_answer(reply)

print(solve_v9(CHALLENGE[0][0], verbose=True))
r9 = evaluate("T9 planner-executor", solve_v9, CHALLENGE)

PLAN:
LOOKUP: The founding year of Indus Robotics is 2015.
LOOKUP: The founding year of Thar Solar is 2016.
COMPUTE: 2016 - 2015 = 1 year
FACTS:
The founding year of Indus Robotics is 2015.: 2009
The founding year of Thar Solar is 2016.: 2014
COMBINER:
CALC: (2016 - 2015) + (2014 - 2009)
6
Q1/3 ✗   Q2/3 ✗   Q3/3 ✗   
EXECUTION REPORT -- T9 planner-executor
Score: 0/3 = 0.00   model calls per question: 4.0   time: 17s
  [wrong ] expected '5'              got '6'
  [wrong ] expected '1960'           got "Error: could not evaluate '10000 - 2300 = 7700' (invalid syntax (<unkn"
  [wrong ] expected '18'             got "Error: could not evaluate '2004 - 1965 = 39' (invalid syntax (<unknown"


**Task 9 questions**
1. Print the plan for one question. Was the plan sensible? Where did your harness fail — planning, lookup, or computing?
2. How many model calls per question does your harness use? Is the extra cost justified?

*Your answers:*

1. The plan for Q1 was only partly sensible. It had the right shape (look up two years, then subtract), but the planner wrote madeup years into the lines ("founding year of Indus Robotics is 2015" and "Thar Solar is 2016"). The lookup step still found the correct years (2009 and 2014) from the passages. The combiner then mixed the madeup and real numbers and wrote (2016-2015)+(2014-2009), which gives 6 instead of 5. So the first failure was in the planner, which then confused the combiner. In Q2 and Q3 the combiner also wrote expressions with an "=" sign (like 10000 - 2300 = 7700), which the calculator cannot read, and used numbers that are not in the passages.
2. It used 4 model calls per question (1 planner, 2 lookups, 1 combiner). With a score of 0/3, the extra cost was not justified in my version. The design is good because it uses retrieval and a calculator together, but the planner is the weak point and needs a stricter prompt

---
## Your score table

Run this cell at the end (wherever you stopped). Copy the table into your lab report.

In [22]:
show_scores()

Harness                                        Score  Calls/Q
T1 single call                                  0.00      1.0
T2 + answer extraction                          0.67      1.0
T3 prompt chain                                 0.50      2.0
T4 agent loop + calculator                      0.25      5.8
T5 code interpreter + error feedback            1.00      1.1
T5b code interpreter, max_tries=1               0.92      1.0
T6a closed book (no retrieval)                  0.00      1.0
T6 retrieve-then-answer                         1.00      1.0
T7 baseline: single-hop RAG on 2-hop Qs         0.17      1.0
T7a multi-hop, summary only                     0.00      3.0
T7b multi-hop, raw passages                     0.33      3.0
T8 multi-hop + verify-then-retry                0.00      3.0
T9 planner-executor                             0.00      4.0
T8 multi-hop + verify-then-retry                0.33      3.0


## Final reflection (everyone answers)
1. Which **single change** to the harness gave the biggest improvement? Why?
2. Which harness would you choose for (a) arithmetic questions, (b) questions about the fictional companies — considering **both** score and model calls?
3. The model was the same in every task. In one or two sentences, what does this lab show about the statement **Agent = Model + Harness**?

*Your answers:*

1. The biggest single improvement was adding retrieval (Task 6a to Task 6), which took the score from 0/6 to 6/6 with no extra calls. The model could not know facts about invented companies, so it needed the passages. For the maths questions, the biggest early jump was adding answer extraction (Task 1 to Task 2), from 0/12 to 8/12 at no extra cost. The model was often right but returned its answer in the wrong form.
2. (a) For arithmetic I would choose Task 5, the code interpreter. It scored 12/12 with only 1.1 calls per question, while Task 2 scored 8/12 with 1.0 call. The agent loop (Task 4) was both worse (3/12) and more expensive (5.8 calls). (b) For questions about the companies I would choose Task 6 (RAG) for direct facts, since it scored 6/6 with 1 call. For two-hop questions the best was Task 7b (2/6, 3 calls). Task 8 gave the same score, so its extra check was not worth adding.
3. The model was exactly the same in every task, yet scores went from 0.00 to 1.00 just by changing the code around it. This shows that how well an agent works depends on both the model and the harness. It also shows that a more complex harness is not always better as the prompt chain,the agent loop and the summary only version were all worse than simpler ones.